## Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_absolute_error

# Setting a theme for our charts to make them look nice
sns.set_theme(style="whitegrid")

In [ ]:
file_path_ground = "../data/ground_truth.csv"
file_path_llama = "../data/ollama_aspects_llama_final.csv"
file_path_gemma = "../data/ollama_aspects_gemma4e4b_final.csv"
file_path_gpt = "../data/groq_aspects_gpt.csv"

df_ground = pd.read_csv(file_path_ground)
df_llama = pd.read_csv(file_path_llama)
df_gemma = pd.read_csv(file_path_gemma)
df_gpt = pd.read_csv(file_path_gpt)

In [ ]:
df_llama = df_llama.rename(columns={
    "overall_score": "overall_llama",
    "food_score": "food_llama",
    "service_score": "service_llama",
    "price_score": "price_llama",
    "ambiance_score": "ambiance_llama"
})

df_gemma = df_gemma.rename(columns={
    "overall_score": "overall_gemma",
    "food_score": "food_gemma",
    "service_score": "service_gemma",
    "price_score": "price_gemma",
    "ambiance_score": "ambiance_gemma",
    "ambiance_quotes": "ambiance_quotes_gemma"
})

df_gpt = df_gpt.rename(columns={
    "overall_score": "overall_gpt",
    "food_score": "food_gpt",
    "service_score": "service_gpt",
    "price_score": "price_gpt",
    "ambiance_score": "ambiance_gpt",
    "ambiance_quotes": "ambiance_quotes_gpt"
    })

In [ ]:
df_llama["review_id"] = df_ground["review_id"]
df_gemma["review_id"] = df_ground["review_id"]
df_gpt["review_id"] = df_ground["review_id"]

In [ ]:
df_merged = df_ground.merge(df_llama, on="review_id", how="left").merge(df_gemma, on="review_id", how="left").merge(df_gpt, on="review_id", how="left")

df_merged

In [ ]:
df_merged = df_merged.rename(columns={
    "ground_truths_ambience": "ground_truths_ambiance"
})

## MAE & SD— LLaMA 3.2 3B

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_llama"

    # Remove rows with missing values
    df_clean = df_merged.dropna(subset=[gt_col, model_col]).copy()

    # Absolute error per row
    df_clean["abs_error"] = abs(df_clean[gt_col] - df_clean[model_col])

    # MAE
    mae = df_clean["abs_error"].mean()

    # Standard deviation of errors
    std_dev = df_clean["abs_error"].std()

    print(f"{aspect.capitalize()}")
    print(f"MAE: {mae:.4f}")
    print(f"Standard Deviation: {std_dev:.4f}")
    print("-" * 30)

## MAE & SD— Gemma 4 9B

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_gemma"

    # Remove rows with missing values
    df_clean = df_merged.dropna(subset=[gt_col, model_col]).copy()

    # Absolute error per row
    df_clean["abs_error"] = abs(df_clean[gt_col] - df_clean[model_col])

    # MAE
    mae = df_clean["abs_error"].mean()

    # Standard deviation of errors
    std_dev = df_clean["abs_error"].std()

    print(f"{aspect.capitalize()}")
    print(f"MAE: {mae:.4f}")
    print(f"Standard Deviation: {std_dev:.4f}")
    print("-" * 30)

## MAE & SD— GPT-OSS-120B

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_gpt"

    # Remove rows with missing values
    df_clean = df_merged.dropna(subset=[gt_col, model_col]).copy()

    # Absolute error per row
    df_clean["abs_error"] = abs(df_clean[gt_col] - df_clean[model_col])

    # MAE
    mae = df_clean["abs_error"].mean()

    # Standard deviation of errors
    std_dev = df_clean["abs_error"].std()

    print(f"{aspect.capitalize()}")
    print(f"MAE: {mae:.4f}")
    print(f"Standard Deviation: {std_dev:.4f}")
    print("-" * 30)

## Recall / Coverage

In [ ]:
total_labels = len(df_merged)

print(f"Total number of Labels: {total_labels}")

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_llama"

    # Only rows where ground truth exists
    df_valid = df_merged[df_merged[gt_col].notna()]

    total_labels = len(df_valid)
    total_predictions = df_valid[model_col].count()

    recall = total_predictions / total_labels

    print(f"{aspect.capitalize()} Recall: {recall:.2%}")

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_gemma"

    # Only rows where ground truth exists
    df_valid = df_merged[df_merged[gt_col].notna()]

    total_labels = len(df_valid)
    total_predictions = df_valid[model_col].count()

    recall = total_predictions / total_labels

    print(f"{aspect.capitalize()} Recall: {recall:.2%}")

In [ ]:
aspects = ["overall", "food", "service", "price", "ambiance"]

for aspect in aspects:
    gt_col = f"ground_truths_{aspect}"
    model_col = f"{aspect}_gpt"

    # Only rows where ground truth exists
    df_valid = df_merged[df_merged[gt_col].notna()]

    total_labels = len(df_valid)
    total_predictions = df_valid[model_col].count()

    recall = total_predictions / total_labels

    print(f"{aspect.capitalize()} Recall: {recall:.2%}")

## Error Autopsy — Ambiance

In [ ]:
df_merged['MAE'] = abs(
    df_merged['ground_truths_ambiance'] - df_merged['ambiance_gemma']
)

df_clean = df_merged.dropna(subset=[
    'ground_truths_ambiance',
    'ambiance_gemma'
])

worst_mistakes = df_clean.sort_values(by='MAE', ascending=False).head(2)

print("--- 🕵️‍♀️ ERROR AUTOPSY REPORT 🕵️‍♀️ ---")

for index, row in worst_mistakes.iterrows():
    print(f"Review ID: {row['review_id']}")
    print(f"Review Text: \"{row['text']}\"")
    print(f"AI Quote:    \"{row['ambiance_quotes_gemma']}\"")
    print(f"Human Score: {row['ground_truths_ambiance']} | AI Score: {row['ambiance_gemma']}")
    print(f"Error (MAE): {row['MAE']:.2f}")
    print("-" * 60)

In [ ]:
df_merged['MAE'] = abs(
    df_merged['ground_truths_ambiance'] - df_merged['ambiance_gpt']
)

df_clean = df_merged.dropna(subset=[
    'ground_truths_ambiance',
    'ambiance_gpt'
])

worst_mistakes = df_clean.sort_values(by='MAE', ascending=False).head(2)

print("--- 🕵️‍♀️ ERROR AUTOPSY REPORT 🕵️‍♀️ ---")

for index, row in worst_mistakes.iterrows():
    print(f"Review ID: {row['review_id']}")
    print(f"Review Text: \"{row['text']}\"")
    print(f"AI Quote:    \"{row['ambiance_quotes_gpt']}\"")
    print(f"Human Score: {row['ground_truths_ambiance']} | AI Score: {row['ambiance_gpt']}")
    print(f"Error (MAE): {row['MAE']:.2f}")
    print("-" * 60)